# Wave 1 — Module 1A: Registration Volatility
**Deteksi indikasi "sembunyiin karyawan" (PDUK) dari perubahan jumlah karyawan terdaftar.**

Requirement yang dicakup (`requirements.md`):
- **#1** Headcount turun drastis **dan** tidak ada catatan resign → *perlu dicek*
- **#1** Riwayat < 3 bulan → status netral `INSUFFICIENT_DATA` (bukan skor)
- **#4** Output diurutkan dari paling berisiko + kolom `reason` (alasan dalam bahasa Indonesia)
- **#5** Modul ini **tidak memvonis**. `FLAGGED` = prioritas untuk dicek, keputusan tetap di pemeriksa
- **#6** Hanya data level perusahaan; kolom data pribadi (nama, NIK) otomatis ditolak

### File yang dibutuhkan modul ini
| File | Wajib? | Dipakai untuk |
|---|---|---|
| `headcount_timeseries.csv` | ✅ wajib | Jumlah karyawan aktif per perusahaan per bulan |
| `resign_records.csv` | opsional (sangat disarankan) | Suppression: drop yang cocok dengan catatan resign tidak di-flag |
| `ground_truth.csv` | opsional | Evaluasi: apakah kasus PDUK yang di-inject ke-flag? |

`payroll_timeseries.csv`, `remittance_timeseries.csv`, `employer_master.csv` **tidak dipakai di Module A**. Itu untuk Module B (gaji) dan C (setoran).

### Cara pakai
1. **Runtime → Run all**.
2. Di sel *Load data*, upload 3 file di atas (atau set `DATA_DIR` ke folder Google Drive).
3. Cek output sel *Deteksi kolom*. Kalau ada kolom yang salah tebak, isi manual di `COLUMN_MAP` lalu jalankan ulang dari sel itu.


In [ ]:
import os, glob
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running di Colab:", IN_COLAB)

## 1. Load data
Pilih salah satu:
- **Opsi A (default):** jalankan sel ini, lalu klik *Choose files* dan pilih `headcount_timeseries.csv`, `resign_records.csv`, `ground_truth.csv` (boleh sekalian semua 6 file).
- **Opsi B (Google Drive):** set `USE_DRIVE = True` dan ubah `DRIVE_DIR` ke folder `Dummy Healthkathon` di Drive kamu. Tidak perlu upload ulang tiap kali runtime restart.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/Dummy Healthkathon"   # ganti sesuai lokasi folder kamu

DATA_DIR = Path(os.environ.get("HK_DATA_DIR", "/content/data"))

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
elif IN_COLAB and not (DATA_DIR / "headcount_timeseries.csv").exists():
    from google.colab import files
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    for name, content in uploaded.items():
        # Colab kadang menambah suffix " (1)" kalau file sudah ada
        clean = name.split(" (")[0] if " (" in name else name
        (DATA_DIR / clean).write_bytes(content)

print("DATA_DIR =", DATA_DIR)
for f in sorted(DATA_DIR.glob("*.csv")):
    print("  ✓", f.name)
assert (DATA_DIR / "headcount_timeseries.csv").exists(), "headcount_timeseries.csv belum ada di DATA_DIR!"


## 2. Intip skema semua file
Sekadar untuk lihat nama kolom dan contoh isi, supaya mapping di langkah berikutnya bisa dicek.

In [ ]:
RAW = {}
for f in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(f)
    RAW[f.stem] = df
    print(f"=== {f.name}  ({len(df):,} baris) ===")
    print(df.dtypes.to_string())
    display(df.head(3)) if "display" in globals() else print(df.head(3))
    print()

## 3. Deteksi kolom (column mapping)
Notebook menebak nama kolom dari daftar kandidat umum. **Kalau tebakan salah, isi manual** di `COLUMN_MAP` (misal `"hc_count": "jumlah_peserta_aktif"`), biarkan `None` untuk auto-detect.

Catatan untuk `resign_records`:
- Kalau ada kolom jumlah (mis. `n_resign`), itu yang dipakai.
- Kalau **satu baris = satu karyawan keluar** (tidak ada kolom jumlah), notebook menghitung jumlah baris per perusahaan per bulan.
- Kalau ada kolom tipe/alasan (mis. `resign`, `mutasi`, `PHK`, `meninggal`), semua dianggap keluar resmi.

In [ ]:
COLUMN_MAP = {
    # headcount_timeseries
    "hc_employer": None,
    "hc_period":   None,
    "hc_count":    None,
    # resign_records
    "rs_employer": None,
    "rs_period":   None,   # kolom bulan ATAU tanggal resign
    "rs_count":    None,   # kosongkan kalau 1 baris = 1 karyawan
    # ground_truth
    "gt_employer": None,
    "gt_label":    None,   # kolom tipe fraud / label
}

# Label ground truth yang dianggap "kasus Module A" (sembunyiin karyawan).
# Kolom label akan dicek apakah MENGANDUNG salah satu string ini (tidak case-sensitive).
POSITIVE_LABELS = ["PDUK", "HIDDEN", "HEADCOUNT", "UNDERREPORT_EMPLOYEE", "SEMBUNYI"]

CANDIDATES = {
    "employer": ["employer_id", "id_employer", "company_id", "perusahaan_id", "id_perusahaan",
                 "badan_usaha_id", "kode_badan_usaha", "kode_bu", "npp", "employer"],
    "period":   ["period", "periode", "month", "bulan", "year_month", "yearmonth", "date",
                 "tanggal", "resign_date", "tanggal_resign", "effective_date", "exit_date", "event_date"],
    "hc_count": ["headcount", "active_headcount", "n_active", "active_employees", "jumlah_karyawan",
                 "jumlah_peserta", "n_employees", "employee_count", "registered_headcount", "hc"],
    "rs_count": ["n_resign", "resign_count", "jumlah_resign", "n_exit", "exit_count", "count",
                 "n_keluar", "jumlah_keluar", "n_terminated", "jumlah"],
    "gt_label": ["fraud_type", "label", "anomaly_type", "injected_fraud", "fraud_label", "scenario",
                 "case_type", "jenis_fraud", "tipe", "is_fraud", "is_pduk", "pduk", "fraud"],
}

def _pick(df, key, override=None, required=True):
    if override:
        if override not in df.columns:
            raise KeyError(f"Kolom '{override}' tidak ada. Kolom tersedia: {list(df.columns)}")
        return override
    lower = {c.lower(): c for c in df.columns}
    for cand in CANDIDATES[key]:
        if cand in lower:
            return lower[cand]
    # fallback: partial match
    for cand in CANDIDATES[key]:
        for lc, orig in lower.items():
            if cand in lc:
                return orig
    if required:
        raise KeyError(f"Tidak bisa menebak kolom '{key}'. Isi manual di COLUMN_MAP. Kolom: {list(df.columns)}")
    return None

hc_raw = RAW["headcount_timeseries"]
rs_raw = RAW.get("resign_records")
gt_raw = RAW.get("ground_truth")

COLS = {
    "hc_employer": _pick(hc_raw, "employer", COLUMN_MAP["hc_employer"]),
    "hc_period":   _pick(hc_raw, "period",   COLUMN_MAP["hc_period"]),
    "hc_count":    _pick(hc_raw, "hc_count", COLUMN_MAP["hc_count"]),
}
if rs_raw is not None:
    COLS["rs_employer"] = _pick(rs_raw, "employer", COLUMN_MAP["rs_employer"])
    COLS["rs_period"]   = _pick(rs_raw, "period",   COLUMN_MAP["rs_period"])
    COLS["rs_count"]    = _pick(rs_raw, "rs_count", COLUMN_MAP["rs_count"], required=False)
if gt_raw is not None:
    COLS["gt_employer"] = _pick(gt_raw, "employer", COLUMN_MAP["gt_employer"])
    COLS["gt_label"]    = _pick(gt_raw, "gt_label", COLUMN_MAP["gt_label"])

print("Mapping yang dipakai:")
for k, v in COLS.items():
    print(f"  {k:12s} -> {v}")
if rs_raw is not None and COLS.get("rs_count") is None:
    print("  (resign_records tidak punya kolom jumlah -> dihitung per baris)")
if rs_raw is None:
    print("  ⚠ resign_records.csv tidak ada -> suppression tidak aktif")

## 4. Konfigurasi (bisa di-tuning)
Setelah sweep di langkah 9, **tulis angka final di sini** sebagai dokumentasi.

In [ ]:
@dataclass(frozen=True)
class ConfigA:
    Z_THRESHOLD: float = 2.0      # flag kalau z < -Z_THRESHOLD
    ROLLING_WINDOW: int = 6       # jumlah periode baseline (pakai sebanyak yang tersedia)
    MIN_PERIODS: int = 3          # cold-start: < 3 periode -> INSUFFICIENT_DATA
    MIN_DROP_PCT: float = 0.10    # drop TAK terjelaskan minimal 10% dari headcount sebelumnya
    STD_FLOOR_ABS: float = 1.0    # std minimum (orang) -> cegah z tak hingga saat seri datar
    STD_FLOOR_REL: float = 0.02   # std minimum relatif thd headcount sebelumnya
    SCORE_CAP_MULT: float = 3.0   # score_a_norm = min(score_a / (Z_THRESHOLD*mult), 1)

CFG = ConfigA()
asdict(CFG)

## 5. Normalisasi input
Semua input diubah ke skema internal: `employer_id, period, headcount, n_resign`. Kolom data pribadi ditolak (requirement #6).

In [ ]:
FORBIDDEN_COLS = {"nama", "name", "nama_karyawan", "employee_name", "nik", "no_kartu",
                  "no_ktp", "ktp", "alamat", "address", "tanggal_lahir", "birth_date"}

def check_privacy(df, name):
    bad = FORBIDDEN_COLS & {c.lower() for c in df.columns}
    if bad:
        raise ValueError(f"{name} berisi kolom data pribadi {bad}. Hapus dulu sebelum dipakai.")

def to_month(s):
    s2 = s.astype(str).str.strip()
    # dukung format '2025-01', '202501', '2025-01-15', '01/2025'
    s2 = s2.where(~s2.str.fullmatch(r"\d{6}"), s2.str[:4] + "-" + s2.str[4:])
    return pd.to_datetime(s2, errors="coerce").dt.to_period("M")

def normalize_headcount(raw):
    check_privacy(raw, "headcount_timeseries")
    df = pd.DataFrame({
        "employer_id": raw[COLS["hc_employer"]].astype(str),
        "period": to_month(raw[COLS["hc_period"]]),
        "headcount": pd.to_numeric(raw[COLS["hc_count"]], errors="coerce"),
    })
    bad = df["period"].isna() | df["headcount"].isna()
    if bad.any():
        print(f"⚠ {bad.sum()} baris headcount dibuang (period/headcount tidak valid)")
    return df[~bad].groupby(["employer_id", "period"], as_index=False)["headcount"].sum()

def normalize_resign(raw):
    if raw is None:
        return None
    # resign_records boleh punya id karyawan, tapi kita tidak menyimpannya: langsung diagregasi
    df = pd.DataFrame({
        "employer_id": raw[COLS["rs_employer"]].astype(str),
        "period": to_month(raw[COLS["rs_period"]]),
    })
    df["n_resign"] = pd.to_numeric(raw[COLS["rs_count"]], errors="coerce").fillna(0) if COLS.get("rs_count") else 1
    df = df.dropna(subset=["period"])
    return df.groupby(["employer_id", "period"], as_index=False)["n_resign"].sum()

HC = normalize_headcount(hc_raw)
RS = normalize_resign(rs_raw)
print(f"Headcount: {HC['employer_id'].nunique():,} employer, {HC['period'].nunique()} periode "
      f"({HC['period'].min()} s/d {HC['period'].max()})")
if RS is not None:
    print(f"Resign   : {len(RS):,} baris agregat, total {int(RS['n_resign'].sum()):,} orang keluar tercatat")
HC.head()

## 6. Skor per periode
Per employer:
- `delta_t = headcount_t − headcount_{t−1}`
- `delta_adj_t = delta_t + (bagian drop yang dijelaskan catatan resign)` → **suppression**
- `mean_delta`, `std_delta` = rolling dari `delta_adj` **periode sebelumnya** (shift 1), supaya drop bulan ini tidak ikut membentuk baseline-nya sendiri
- `z_t = (delta_adj_t − mean_delta) / std_delta`
- **FLAG periode** kalau `z_t < −Z_THRESHOLD` **dan** drop tak terjelaskan ≥ `MIN_DROP_PCT`

In [ ]:
def compute_period_scores(hc, rs, cfg=CFG):
    df = hc.copy()
    df = df.merge(rs, on=["employer_id", "period"], how="left") if rs is not None else df.assign(n_resign=0)
    df["n_resign"] = df["n_resign"].fillna(0)
    df = df.sort_values(["employer_id", "period"]).reset_index(drop=True)

    g = df.groupby("employer_id", sort=False)
    df["n_periods"] = g["period"].transform("count")
    df["prev_hc"] = g["headcount"].shift(1)
    df["delta"] = df["headcount"] - df["prev_hc"]

    # Suppression: resign hanya menjelaskan PENURUNAN, maksimal sebesar dropnya
    drop = (-df["delta"]).clip(lower=0)
    df["explained"] = np.minimum(df["n_resign"], drop)
    df["delta_adj"] = df["delta"] + df["explained"]

    W = cfg.ROLLING_WINDOW
    grp = df.groupby("employer_id", sort=False)["delta_adj"]
    df["mean_delta"] = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=1).mean())
    std_hist = grp.transform(lambda s: s.shift(1).rolling(W, min_periods=2).std())
    floor = np.maximum(cfg.STD_FLOOR_ABS, cfg.STD_FLOOR_REL * df["prev_hc"])
    df["std_delta"] = np.maximum(std_hist.fillna(0), floor)

    df["z_raw"] = (df["delta"] - df["mean_delta"]) / df["std_delta"]
    df["z"] = (df["delta_adj"] - df["mean_delta"]) / df["std_delta"]
    df["drop_pct"] = drop / df["prev_hc"]
    df["unexplained_pct"] = (-df["delta_adj"]).clip(lower=0) / df["prev_hc"]

    has_base = df["mean_delta"].notna() & (df["n_periods"] >= cfg.MIN_PERIODS)
    thr = cfg.Z_THRESHOLD
    df["flag_raw"] = has_base & (df["z_raw"] < -thr) & (df["drop_pct"] >= cfg.MIN_DROP_PCT)
    df["flag"] = has_base & (df["z"] < -thr) & (df["unexplained_pct"] >= cfg.MIN_DROP_PCT)
    df["suppressed"] = df["flag_raw"] & ~df["flag"]

    sev = (-df["z"]).clip(lower=0)
    df["severity"] = np.where(has_base & (df["unexplained_pct"] >= cfg.MIN_DROP_PCT), sev, 0.0)
    return df

PERIODS = compute_period_scores(HC, RS)
PERIODS[PERIODS["flag"]].head()

## 7. Agregasi ke level employer + alasan

In [ ]:
FLAGGED, EXPLAINED, NORMAL, INSUFFICIENT = "FLAGGED", "EXPLAINED_BY_RESIGN", "NORMAL", "INSUFFICIENT_DATA"

def make_reason(r, cfg=CFG):
    if r["status"] == INSUFFICIENT:
        return f"Riwayat data baru {int(r['n_periods'])} bulan (< {cfg.MIN_PERIODS}); belum bisa dinilai."
    if r["status"] == FLAGGED:
        extra = (f"; hanya {int(r['resign_recorded'])} orang tercatat resign" if r["resign_recorded"] > 0
                 else "; tidak ada catatan resign")
        return (f"Perlu dicek — indikasi sembunyiin karyawan: jumlah karyawan turun {int(r['drop'])} orang "
                f"({r['drop_pct']:.0%}) pada {r['worst_period']} ({int(r['hc_before'])}→{int(r['hc_after'])}), "
                f"jauh di luar pola biasanya (z={r['z_score']:.1f}){extra}.")
    if r["status"] == EXPLAINED:
        return (f"Ada penurunan {int(r['drop'])} orang pada {r['worst_period']}, tetapi sesuai catatan "
                f"resign/mutasi resmi ({int(r['resign_recorded'])} orang) — dianggap wajar.")
    return "Pergerakan jumlah karyawan dalam batas wajar."

def score_employers(periods, cfg=CFG):
    rows = []
    for eid, d in periods.groupby("employer_id", sort=False):
        n_p = int(d["n_periods"].iloc[0])
        base = dict(employer_id=eid, n_periods=n_p)
        if n_p < cfg.MIN_PERIODS:
            rows.append({**base, "status": INSUFFICIENT, "score_a": np.nan, "score_a_norm": np.nan,
                         "n_flagged_periods": 0})
            continue
        d2 = d.dropna(subset=["z"])
        w = d2.sort_values(["flag", "severity", "z"], ascending=[False, False, True]).iloc[0]
        if d["flag"].any():
            status = FLAGGED
        elif d["suppressed"].any():
            status = EXPLAINED
            w = d[d["suppressed"]].sort_values("z_raw").iloc[0]
        else:
            status = NORMAL
        score = float(d["severity"].max())
        rows.append({**base, "status": status, "score_a": round(score, 3),
                     "score_a_norm": round(min(score / (cfg.Z_THRESHOLD * cfg.SCORE_CAP_MULT), 1.0), 3),
                     "n_flagged_periods": int(d["flag"].sum()), "worst_period": str(w["period"]),
                     "hc_before": w["prev_hc"], "hc_after": w["headcount"],
                     "drop": max(-w["delta"], 0), "drop_pct": w["drop_pct"],
                     "resign_recorded": w["n_resign"], "z_score": round(float(w["z"]), 2)})
    out = pd.DataFrame(rows)
    out["reason"] = out.apply(lambda r: make_reason(r, cfg), axis=1)
    order = {FLAGGED: 0, NORMAL: 1, EXPLAINED: 1, INSUFFICIENT: 2}
    out = (out.assign(_o=out["status"].map(order))
              .sort_values(["_o", "score_a"], ascending=[True, False], na_position="last")
              .drop(columns="_o").reset_index(drop=True))
    out.insert(0, "rank", range(1, len(out) + 1))
    return out

def run_module_a(hc, rs, cfg=CFG):
    p = compute_period_scores(hc, rs, cfg)
    return score_employers(p, cfg), p

RESULT_A, PERIODS = run_module_a(HC, RS, CFG)
print(RESULT_A["status"].value_counts().to_string())
RESULT_A.head(15)[["rank", "employer_id", "status", "score_a", "score_a_norm", "reason"]]

## 8. Evaluasi vs `ground_truth.csv`
- **Recall**: dari semua kasus PDUK yang di-inject, berapa persen yang ke-flag?
- **False positive rate**: dari employer bersih, berapa persen yang salah ke-flag? (harus kecil → *tidak ke-flag berlebihan*)
- Employer `INSUFFICIENT_DATA` dikeluarkan dari hitungan (memang netral by design).

⚠ Kalau ground truth berisi **semua jenis fraud** (termasuk lapor gaji rendah & tidak setor), pastikan `POSITIVE_LABELS` hanya menangkap kasus sembunyiin karyawan. Kasus jenis lain memang bukan tugas Module A.

In [ ]:
def build_truth(gt_raw):
    if gt_raw is None:
        return None
    col = gt_raw[COLS["gt_label"]]
    if col.dtype == bool or set(col.dropna().astype(str).str.lower().unique()) <= {"0", "1", "true", "false"}:
        pos = col.astype(str).str.lower().isin(["1", "true"])
        print(f"⚠ Kolom '{COLS['gt_label']}' berupa boolean -> semua fraud dianggap positif untuk Module A")
    else:
        pat = "|".join(POSITIVE_LABELS)
        pos = col.astype(str).str.upper().str.contains(pat, na=False)
        print("Nilai label di ground truth:", col.astype(str).value_counts().to_dict())
    t = pd.DataFrame({"employer_id": gt_raw[COLS["gt_employer"]].astype(str), "actual": pos})
    return t.groupby("employer_id", as_index=False)["actual"].any()   # kalau GT per-periode

def evaluate(result, truth, verbose=True):
    m = result.merge(truth, on="employer_id", how="left")
    m["actual"] = m["actual"].fillna(False).astype(bool)
    ev = m[m["status"] != INSUFFICIENT]
    pred = ev["status"] == FLAGGED
    tp, fp = int((pred & ev.actual).sum()), int((pred & ~ev.actual).sum())
    fn, tn = int((~pred & ev.actual).sum()), int((~pred & ~ev.actual).sum())
    res = dict(tp=tp, fp=fp, fn=fn, tn=tn,
               precision=round(tp / (tp + fp), 3) if tp + fp else np.nan,
               recall=round(tp / (tp + fn), 3) if tp + fn else np.nan,
               false_positive_rate=round(fp / (fp + tn), 3) if fp + tn else np.nan,
               n_insufficient=int((m.status == INSUFFICIENT).sum()),
               positif_di_insufficient=int(m.loc[m.status == INSUFFICIENT, "actual"].sum()))
    if verbose:
        for k, v in res.items():
            print(f"  {k:24s}: {v}")
        missed = ev[~pred & ev.actual]
        fa = ev[pred & ~ev.actual]
        if len(missed):
            print("\nKasus PDUK yang TERLEWAT:")
            print(missed[["employer_id", "status", "score_a", "z_score", "drop_pct", "reason"]].to_string(index=False))
        if len(fa):
            print("\nEmployer bersih yang SALAH ke-flag:")
            print(fa[["employer_id", "score_a", "z_score", "drop_pct", "reason"]].to_string(index=False))
    return res

TRUTH = build_truth(gt_raw)
if TRUTH is not None:
    print(f"Positif (kasus Module A) di ground truth: {int(TRUTH.actual.sum())} employer\n")
    EVAL_A = evaluate(RESULT_A, TRUTH)
else:
    print("ground_truth.csv tidak ada — evaluasi dilewati")

## 9. Tuning threshold (sweep)
Pilih kombinasi dengan **recall tinggi** tapi **false positive rate rendah**. Setelah itu, update `ConfigA` di langkah 4 lalu *Run all* lagi.

In [ ]:
def threshold_sweep(z_grid=(1.5, 2.0, 2.5, 3.0, 4.0), drop_grid=(0.05, 0.10, 0.15, 0.20)):
    rows = []
    for z in z_grid:
        for dp in drop_grid:
            cfg = replace(CFG, Z_THRESHOLD=z, MIN_DROP_PCT=dp)
            res, _ = run_module_a(HC, RS, cfg)
            e = evaluate(res, TRUTH, verbose=False)
            rows.append(dict(Z_THRESHOLD=z, MIN_DROP_PCT=dp,
                             **{k: e[k] for k in ("tp", "fp", "fn", "precision", "recall", "false_positive_rate")}))
    return pd.DataFrame(rows)

if TRUTH is not None:
    SWEEP = threshold_sweep()
    display(SWEEP) if "display" in globals() else print(SWEEP.to_string(index=False))

## 10. Visual sanity check
Beberapa employer ter-flag (merah = periode yang di-flag, oranye = drop yang dijelaskan resign).

In [ ]:
def plot_employers(ids, periods=PERIODS):
    ids = list(ids)
    if not ids:
        print("Tidak ada employer untuk di-plot"); return
    n = len(ids)
    fig, axes = plt.subplots(n, 1, figsize=(10, 2.3 * n), squeeze=False)
    for ax, eid in zip(axes[:, 0], ids):
        d = periods[periods.employer_id == eid]
        x = d["period"].astype(str)
        ax.plot(x, d["headcount"], marker="o", color="steelblue")
        ax.scatter(x[d["flag"]], d.loc[d["flag"], "headcount"], color="red", s=90, zorder=3, label="FLAG")
        ax.scatter(x[d["suppressed"]], d.loc[d["suppressed"], "headcount"], color="orange", s=90, zorder=3,
                   label="dijelaskan resign")
        ax.set_title(eid, fontsize=10, loc="left"); ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.legend(fontsize=7, loc="upper right")
    plt.tight_layout(); plt.show()

plot_employers(RESULT_A.loc[RESULT_A.status == FLAGGED, "employer_id"].head(4))
plot_employers(RESULT_A.loc[RESULT_A.status == EXPLAINED, "employer_id"].head(2))

## 11. Simpan output
`module_a_scores.csv` inilah yang nanti digabung dengan Module B & C (kolom `score_a_norm` sudah 0–1).

In [ ]:
OUT_DIR = Path("output_module_a"); OUT_DIR.mkdir(exist_ok=True)
RESULT_A.to_csv(OUT_DIR / "module_a_scores.csv", index=False)
PERIODS.drop(columns=["n_resign"]).assign(period=PERIODS["period"].astype(str)) \
       .to_csv(OUT_DIR / "module_a_period_detail.csv", index=False)
if TRUTH is not None:
    SWEEP.to_csv(OUT_DIR / "module_a_threshold_sweep.csv", index=False)
print("Tersimpan:", [f.name for f in OUT_DIR.glob("*.csv")])

if IN_COLAB:
    from google.colab import files
    files.download(str(OUT_DIR / "module_a_scores.csv"))